In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import math

In [2]:
clinical_data = pd.read_csv("datasets/clinical_data.csv")
lab_data1 = pd.read_csv("datasets/lab_data.csv")

In [3]:
preop = lab_data1[(lab_data1['dt'] < 0) & (lab_data1['dt'] >= -2592000)] #30 days

preop_last = (preop.sort_values('dt').groupby(['caseid', 'name']).last()   
              .reset_index()[['caseid','dt', 'name', 'result']])

preop_last.head(10)
print("pre-op rows:", len(preop))

pre-op rows: 196862


In [4]:
preop.groupby("name")["dt"].describe()
print("median hours before surgery:", (preop_last["dt"].abs()/3600).median())

median hours before surgery: 29.634444444444444


In [5]:
lab_data = (preop_last
            .pivot(index="caseid", columns="name", values="result")
            .reset_index())
lab_data.columns.name = None

lab_cols = [c for c in lab_data.columns if c != "caseid"]
lab_data[lab_cols] = lab_data[lab_cols].apply(pd.to_numeric, errors="coerce")

lab_data.head(10)
print(lab_data.shape)
print(clinical_data.shape)

(5388, 35)
(6388, 74)


In [6]:
missing_rate = lab_data.isna().mean()
lab_data = lab_data.loc[:, missing_rate < 0.75]
print("after 75% filter:", lab_data.shape)
lab_data


after 75% filter: (5388, 24)


,caseid,alb,alt,aptt,ast,bun,cl,cr,crp,fib,...,k,na,p,plt,pt%,ptinr,ptsec,tbil,tprot,wbc
0,1,NaN,NaN,NaN,NaN,NaN,100.0,NaN,NaN,NaN,...,3.5,141.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,NaN,NaN,NaN,NaN,NaN,115.0,NaN,NaN,NaN,...,4.7,143.0,NaN,251.0,NaN,NaN,NaN,NaN,NaN,4.36
2,4,NaN,NaN,NaN,NaN,NaN,108.0,NaN,NaN,NaN,...,4.3,144.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5,4.6,30.0,36.5,64.0,27.0,108.0,1.83,NaN,178.0,...,4.9,139.0,5.4,158.0,81.0,1.14,13.0,0.9,8.2,14.64
4,6,3.9,17.0,29.1,19.0,14.0,104.0,1.07,0.01,183.0,...,3.4,141.0,2.8,145.0,104.0,0.97,11.1,1.0,6.7,3.20
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5383,6384,NaN,NaN,NaN,NaN,NaN,107.0,NaN,NaN,NaN,...,4.2,143.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5384,6385,3.7,28.0,NaN,18.0,19.0,109.0,0.84,NaN,NaN,...,4.0,144.0,NaN,239.0,NaN,NaN,NaN,0.6,5.9,5.45
5385,6386,NaN,NaN,NaN,NaN,NaN,107.0,NaN,NaN,NaN,...,4.5,142.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5386,6387,4.4,10.0,32.2,16.0,7.0,103.0,0.65,0.03,239.0,...,3.6,142.0,3.0,214.0,90.0,1.07,12.2,0.3,6.9,7.70


In [7]:
# remove physiologically impossible values only
lab_data = lab_data[lab_data["hct"].between(10, 65) | lab_data["hct"].isna()]
for c in ["height", "weight", "bmi"]:
    clinical_data.loc[clinical_data[c] <= 0, c] = np.nan
print("lab_data after plausibility filter:", lab_data.shape)
print("clinical_data after plausibility filter:", clinical_data.shape)

lab_data after plausibility filter: (5388, 24)
clinical_data after plausibility filter: (6388, 74)


start building the decision tree

In [10]:
y = lab_data["hct"]
x = clinical_data[["age","sex","height", "weight", "bmi"]]